# Model registry: which digit classifier should we promote?

Your group serves one model, a small neural network that reads handwritten digits. A teammate's new version reports 1.000 and asks to go live. You need to decide what's in production, say why, and be able to get any released version back a year from now.

A model registry answers two questions: which versions exist, and which one is live. You'll use `gto`, which keeps the registry in Git tags, so there's no server to run. DVC keeps the model files. The model file is cattle, because you can rebuild it. The record of what you released and why is the pet.

**The short path takes about 15 minutes:** do step 1, then run its check. Steps 2 and 3 are optional and take about 25 more. Only reading? Each step shows the failure and says how to approach the fix.

## Get a terminal

You read this page and paste commands into a terminal. You don't run anything on the page. The output shown here is from a real run of these same commands on the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-03-registry?urlpath=lab/workspaces/exercise).** It's free and there's nothing to install. The first start can take a few minutes. You get a terminal on the left, already in the `project` folder, and this page on the right. To copy a command, select it and press Ctrl+C, or Cmd+C on a Mac. To paste it in the terminal, press Ctrl+Shift+V, or Cmd+V on a Mac, or right-click and choose Paste. To edit a file, double-click it in the file panel on the far left, and save with Ctrl+S. A Binder session is temporary and has 2 GB of memory, so download anything you want to keep.

**Or use your laptop's terminal,** with this page open in your browser. It works on Linux and macOS, on x86_64 or ARM, and on Windows through WSL2. You need [conda](https://github.com/conda-forge/miniforge), and you don't need Jupyter. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`. Paste this once:

```bash
git clone --branch binder-03-registry --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git registry
cd registry
conda env create --quiet --file .binder/environment.yml
conda activate repro-registry
python scripts/setup_project.py
cd project
```

Every command below runs in that `project` folder. To start over, go to the folder above it and run `python scripts/setup_project.py --reset`, which deletes your work.

## The first release

`project` already has a pipeline, a first release in prod, and a branch from your teammate Sam. `../osn-bucket` stands in for an object storage bucket such as OSN.

The width of the network is in `params.yaml`, and `dvc repro` trains it. `split.py` writes the lists of training, validation and test rows to `splits.json`, and Git keeps that file. `train.py` reports accuracy on the validation rows only.

These are the commands that made the first release. `gto register` gives a commit a version. `gto assign` says which version is in which stage. Both only make Git tags. You don't need to run these.

```text
dvc repro
git add .
git commit -m "Train with 32 hidden units"
gto register digits-clf --version v1.0.0
gto assign digits-clf --version v1.0.0 --stage prod
```

Look at the registry. `score.py` doesn't use the model in your folder. It asks the registry what's in prod, gets that version, and scores it on the test rows. That's the only place the test rows get used.

```bash
cat params.yaml
gto show
python score.py
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import run

run("cat params.yaml", "gto show", "python score.py")

$ cat params.yaml
split:
  seed: 2026
train:
  # Hidden units in the network's one hidden layer.
  hidden: 32

$ gto show
╒════════════╤══════════╤═════════╕
│ name       │ latest   │ #prod   │
╞════════════╪══════════╪═════════╡
│ digits-clf │ v1.0.0   │ v1.0.0  │
╘════════════╧══════════╧═════════╛

$ python score.py
prod is v1.0.0. Accuracy on the 360 test rows: 0.978


## Step 1: train another width, and decide

Is another width better? Pick one yourself, and don't assume bigger wins. Train it, register it as a new version, and decide which version goes to prod.

Decide on validation accuracy only. If you pick the model using the test rows, the test score stops being an honest estimate of how it will do on digits it has never seen. So only compare `val_accuracy`.

Your turn:

1. Set `hidden` in `params.yaml` to your width, and save it. Then train and compare.

```bash
dvc repro
dvc metrics diff digits-clf@v1.0.0
```

2. Commit it and register it as a new version.

```bash
git add .
git commit --quiet -m "Train with a new width"
gto register digits-clf --version v1.1.0
```

3. Decide. If your new width won, promote it with this. If it didn't, skip this line, because prod is already right. That's a good outcome.

```bash
gto assign digits-clf --version v1.1.0 --stage prod
```

4. Push, and look at the record.

```bash
dvc push
gto show
gto history digits-clf
python score.py
```

### Check the short path

Run this when step 1 is done. It looks at every version you registered, works out which has the best validation accuracy, then clones your project into an empty folder and rebuilds prod from scratch. Each line says what to do next. When it passes, your prod version is your best honest model and a stranger can serve it. You can stop here.

```bash
python ../check.py --short
```

On the starter project it fails like this:

In [2]:
# Recorded output. You don't need to run this.
run("python ../check.py --short")

$ python ../check.py --short
FAIL  Register at least two versions. Train a second width, commit it, and use gto register (step 1).
PASS  Prod is the version with the best validation accuracy
FAIL  A fresh clone can't serve prod, which is v1.0.0. Its model isn't in the bucket. In project/, run git checkout digits-clf@v1.0.0, then dvc push, then git checkout main.

1 of 3 parts pass.


## Step 2 (optional): a promotion you should refuse

Sam wants a promotion. Here's the request, and how the candidate compares with the first release.

```bash
git show candidate-v2:PROMOTION_REQUEST.md
dvc metrics diff digits-clf@v1.0.0 candidate-v2
```

In [3]:
# Recorded output. You don't need to run this.
run("git show candidate-v2:PROMOTION_REQUEST.md", "dvc metrics diff digits-clf@v1.0.0 candidate-v2")

$ git show candidate-v2:PROMOTION_REQUEST.md
# Please promote candidate-v2

Validation accuracy is 1.000, up from what we have in prod. I also gave the
training step more data, which always helps. Can you register this and move it
to prod before the review?

Sam

$ dvc metrics diff digits-clf@v1.0.0 candidate-v2
Path          Metric        digits-clf@v1.0.0    candidate-v2    Change
metrics.json  val_accuracy  0.978                1.0             0.022


A classifier that never gets a handwritten digit wrong should make you suspicious. A perfect score can mean the model has seen the answers. The pipeline ran without an error, and the score is real, so no tool will flag this for you. You need a test.

`splits.json` holds three lists of row numbers, under `train`, `val` and `test`. Sam's branch has its own copy.

Your turn:

1. Write `tests/test_split.py`. It should read `splits.json` from the project folder and fail if any row is in more than one list.
2. Run it on your branch, where it should pass. Run it on Sam's, where it should fail. The test file isn't committed yet, so it stays in your folder when you switch branches.

```bash
python -m pytest -q tests/test_split.py
git checkout --quiet candidate-v2
python -m pytest -q tests/test_split.py
git checkout --quiet main
```

3. Write `decisions.md`. Refuse the promotion and say what's wrong, so that nobody has to find it again.
4. Commit both.

```bash
git add tests decisions.md
git commit --quiet -m "Test that the splits don't overlap, and refuse candidate-v2"
```

## Step 3 (optional): get an old release back

A year from now, someone asks you to rerun the model behind an old result. This script acts as that person. It clones your project into an empty folder, so it has an empty cache, and tries to restore every registered version by its tag.

```bash
python ../scripts/stranger.py
```

In [4]:
# Recorded output. You don't need to run this.
run("python ../scripts/stranger.py")

$ python ../scripts/stranger.py
digits-clf@v1.0.0: FAILED. Its model isn't in the bucket, so a stranger can't get it.


The tag is there and the pointer is there, but the model isn't in the bucket. Nobody ran `dvc push` for that release. A plain `dvc push` only sends what your current files point to, so pushing later doesn't help the earlier version. The model is only in this folder's cache, and when this folder goes, the release goes with it.

Your turn. Get every registered version into the bucket, then run `stranger.py` again until each one restores. The `--help` for `dvc push` will show you more than one way to do it.

```bash
dvc push --help
```

## Check everything

This is the full check, for steps 1 to 3. It gives your split test Sam's splits and some bad splits of its own, and it restores each version into an empty folder. It works in throwaway clones, so your project is safe.

```bash
python ../check.py
```

On the starter project it fails like this:

In [5]:
# Recorded output. You don't need to run this.
run("python ../check.py")

$ python ../check.py
FAIL  Register at least two versions. Train a second width, commit it, and use gto register (step 1).
PASS  Prod is the version with the best validation accuracy
FAIL  A fresh clone can't serve prod, which is v1.0.0. Its model isn't in the bucket. In project/, run git checkout digits-clf@v1.0.0, then dvc push, then git checkout main.
FAIL  Commit tests/test_split.py, which reads splits.json and tests that the row lists don't overlap (step 2).
FAIL  Commit decisions.md. Name candidate-v2 and say what's wrong with its rows, so the next person doesn't have to find it again.
FAIL  These versions aren't in the bucket: v1.0.0. See step 3.

1 of 6 parts pass.


## On your own project

Think of the last model you shared. Could you get that exact file back today, from a clean machine, knowing only its version? Could someone else tell why it was chosen over the others? If not, register it, push it, and write down the decision. Then write the one test that would have caught your worst mistake.

## About the data

The Optical Recognition of Handwritten Digits data: 1,797 images of 8 by 8 pixels. It's included in scikit-learn as `load_digits`, so there's nothing to download. The UCI Machine Learning Repository publishes the data under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), and scikit-learn's code is BSD licensed.

Cite it as: Alpaydin, E. and Kaynak, C. (1998). Optical Recognition of Handwritten Digits. UCI Machine Learning Repository. doi:10.24432/C50P49.